# 92. The Location-Routing Problem (LRP)
## Tier 6: The Autonomous & Self-Optimizing Ecosystem (Multi-Agent System)

### Key assumptions
- Multi-agent system with decentralized intelligence and decision-making
- Customer agents broadcast demand and service requirements
- Depot agents bid for customer assignments based on cost and capability
- Vehicle agents negotiate to form efficient routes through auction mechanisms
- Contract Net Protocol enables agent coordination and negotiation
- Emergent behavior creates optimal solutions without central control

### Approach (step-by-step)
1. **Agent Architecture**: Define customer, depot, and vehicle agent behaviors
2. **Communication Protocol**: Implement Contract Net Protocol for agent interactions
3. **Auction Mechanism**: Design bidding and negotiation processes
4. **Emergent Optimization**: Allow solutions to emerge from agent interactions
5. **System Resilience**: Test robustness under agent failures and disruptions
6. **Performance Analysis**: Compare with centralized optimization approaches

### What to look for in the results
- Emergent depot location decisions from agent bidding
- Self-organizing vehicle routes through negotiation
- System resilience under agent failures or disruptions
- Scalability performance as agent count increases
- Comparison of solution quality vs centralized methods

### Concrete example (from the source)
- **Problem**: 2-depot, 3-customer system with 6 agents total
- **Agents**: 3 customer agents, 2 depot agents, 3 vehicle agents
- **Protocol**: Contract Net Protocol with sealed-bid auctions
- **Expected Behavior**: Emergent optimal configuration through distributed intelligence

In [1]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import time
import math
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional, Set
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

In [ ]:
class MessageType(Enum):
    """Types of messages between agents"""
    ANNOUNCE_DEMAND = "announce_demand"
    REQUEST_BIDS = "request_bids"
    SUBMIT_BID = "submit_bid"
    AWARD_CONTRACT = "award_contract"
    REJECT_BID = "reject_bid"
    REQUEST_ROUTE = "request_route"
    PROPOSE_ROUTE = "propose_route"
    ACCEPT_ROUTE = "accept_route"
    NEGOTIATE_ROUTE = "negotiate_route"
    CONFIRM_ALLOCATION = "confirm_allocation"

@dataclass
class Message:
    """Message passed between agents"""
    sender_id: str
    receiver_id: str
    message_type: MessageType
    content: Dict
    timestamp: float
    priority: int = 1

@dataclass
class Bid:
    """Bid submitted by an agent"""
    agent_id: str
    bid_value: float
    capacity: float
    quality_score: float
    additional_info: Dict
    timestamp: float

@dataclass
class Contract:
    """Contract between agents"""
    contract_id: str
    client_id: str
    provider_id: str
    service_type: str
    terms: Dict
    status: str = "pending"
    performance: Dict = None

@dataclass
class LRPInstance:
    """Data class for Location-Routing Problem instance"""
    customers: List[int]
    depots: List[int]
    vehicles: List[int]
    depot_costs: Dict[int, float]
    demands: Dict[int, float]
    vehicle_capacity: float
    travel_costs: Dict[Tuple[int, int], float]
    
    def get_all_nodes(self):
        return self.customers + self.depots

# Create the concrete example
def create_concrete_example():
    customers = [1, 2, 3]
    depots = [4, 5]
    vehicles = [1, 2, 3]
    
    depot_costs = {4: 100, 5: 120}
    demands = {1: 10, 2: 15, 3: 20}
    vehicle_capacity = 30
    
    travel_costs = {
        (1, 2): 15, (2, 1): 15,
        (1, 3): 20, (3, 1): 20,
        (2, 3): 18, (3, 2): 18,
        (4, 1): 12, (1, 4): 12,
        (4, 2): 14, (2, 4): 14,
        (4, 3): 25, (3, 4): 25,
        (5, 1): 18, (1, 5): 18,
        (5, 2): 16, (2, 5): 16,
        (5, 3): 22, (3, 5): 22,
        (4, 5): 30, (5, 4): 30,
        (1, 1): 0, (2, 2): 0, (3, 3): 0,
        (4, 4): 0, (5, 5): 0
    }
    
    return LRPInstance(
        customers=customers,
        depots=depots,
        vehicles=vehicles,
        depot_costs=depot_costs,
        demands=demands,
        vehicle_capacity=vehicle_capacity,
        travel_costs=travel_costs
    )

instance = create_concrete_example()
print(f"LRP Instance created for Multi-Agent System:")
print(f"Customers: {instance.customers}")
print(f"Potential Depots: {instance.depots}")
print(f"Vehicles: {instance.vehicles}")
print(f"Vehicle Capacity: {instance.vehicle_capacity}")

In [ ]:
class Agent:
    """Base class for all agents in the multi-agent system"""
    
    def __init__(self, agent_id: str, agent_type: str, instance: LRPInstance):
        self.agent_id = agent_id
        self.agent_type = agent_type
        self.instance = instance
        self.message_queue = []
        self.knowledge_base = {}
        self.contracts = []
        self.performance_history = []
        self.active = True
        
    def send_message(self, receiver_id: str, message_type: MessageType, content: Dict, priority: int = 1):
        """Send a message to another agent"""
        message = Message(
            sender_id=self.agent_id,
            receiver_id=receiver_id,
            message_type=message_type,
            content=content,
            timestamp=time.time(),
            priority=priority
        )
        return message
    
    def receive_message(self, message: Message):
        """Receive and process a message"""
        self.message_queue.append(message)
        self.message_queue.sort(key=lambda m: m.priority, reverse=True)
    
    def process_messages(self):
        """Process all pending messages"""
        while self.message_queue:
            message = self.message_queue.pop(0)
            self.handle_message(message)
    
    def handle_message(self, message: Message):
        """Handle a specific message - to be overridden by subclasses"""
        pass
    
    def update_knowledge(self, key: str, value):
        """Update the agent's knowledge base"""
        self.knowledge_base[key] = {
            'value': value,
            'timestamp': time.time(),
            'source': 'self'
        }

class CustomerAgent(Agent):
    """Agent representing a customer with demand"""
    
    def __init__(self, agent_id: str, customer_id: int, instance: LRPInstance):
        super().__init__(agent_id, "customer", instance)
        self.customer_id = customer_id
        self.demand = instance.demands[customer_id]
        self.location = customer_id
        self.service_requirements = {
            'demand': self.demand,
            'location': self.location,
            'service_level': 0.95,
            'time_window': (8, 18),  # 8 AM to 6 PM
            'priority': 'normal'
        }
        self.assigned_depot = None
        self.assigned_vehicle = None
        self.contract_status = "unassigned"
    
    def announce_demand(self, depot_agents: List['DepotAgent']):
        """Announce demand to all depot agents"""
        messages = []
        for depot_agent in depot_agents:
            if depot_agent.active:
                message = self.send_message(
                    receiver_id=depot_agent.agent_id,
                    message_type=MessageType.ANNOUNCE_DEMAND,
                    content=self.service_requirements,
                    priority=2
                )
                messages.append(message)
        return messages
    
    def evaluate_bids(self, bids: List[Bid]) -> Optional[Bid]:
        """Evaluate received bids and select the best one"""
        if not bids:
            return None
        
        # Score bids based on cost, quality, and capacity
        scored_bids = []
        for bid in bids:
            # Lower cost is better (inverse scoring)
            cost_score = 1.0 / (1.0 + bid.bid_value)
            
            # Higher quality and capacity are better
            quality_score = bid.quality_score
            capacity_score = min(1.0, bid.capacity / self.demand)
            
            # Weighted score
            total_score = 0.4 * cost_score + 0.3 * quality_score + 0.3 * capacity_score
            scored_bids.append((bid, total_score))
        
        # Select best bid
        scored_bids.sort(key=lambda x: x[1], reverse=True)
        return scored_bids[0][0]
    
    def handle_message(self, message: Message):
        """Handle messages from other agents"""
        if message.message_type == MessageType.AWARD_CONTRACT:
            # Depot has awarded service contract
            self.assigned_depot = message.content['depot_id']
            self.contract_status = "assigned"
            self.update_knowledge('assigned_depot', self.assigned_depot)
            
        elif message.message_type == MessageType.REJECT_BID:
            # Bid was rejected, may need to re-announce demand
            self.contract_status = "rejected"
            
        elif message.message_type == MessageType.CONFIRM_ALLOCATION:
            # Vehicle allocation confirmed
            self.assigned_vehicle = message.content['vehicle_id']
            self.update_knowledge('assigned_vehicle', self.assigned_vehicle)

class DepotAgent(Agent):
    """Agent representing a potential depot location"""
    
    def __init__(self, agent_id: str, depot_id: int, instance: LRPInstance):
        super().__init__(agent_id, "depot", instance)
        self.depot_id = depot_id
        self.location = depot_id
        self.fixed_cost = instance.depot_costs[depot_id]
        self.capacity = float('inf')  # Unlimited capacity for simplicity
        self.assigned_customers = []
        self.assigned_vehicles = []
        self.is_open = False
        self.total_revenue = 0.0
        self.operating_cost = 0.0
    
    def calculate_bid(self, customer_requirements: Dict) -> Bid:
        """Calculate a bid for serving a customer"""
        customer_demand = customer_requirements['demand']
        customer_location = customer_requirements['location']
        
        # Calculate routing cost to customer
        routing_cost = self.instance.travel_costs.get((self.depot_id, customer_location), float('inf'))
        
        # Calculate total cost (fixed cost amortized + routing cost)
        if not self.is_open:
            # If not open, need to amortize fixed cost
            num_customers_estimate = len(self.assigned_customers) + 3  # Estimate future customers
            amortized_fixed = self.fixed_cost / max(1, num_customers_estimate)
        else:
            amortized_fixed = 0  # Fixed cost already being paid
        
        total_cost = amortized_fixed + routing_cost
        
        # Calculate quality score based on distance and capacity
        distance = routing_cost
        quality_score = 1.0 / (1.0 + distance / 10.0)  # Normalize distance
        
        # Available capacity
        available_capacity = self.capacity - sum(self.instance.demands.get(c, 0) for c in self.assigned_customers)
        
        return Bid(
            agent_id=self.agent_id,
            bid_value=total_cost,
            capacity=available_capacity,
            quality_score=quality_score,
            additional_info={
                'routing_cost': routing_cost,
                'fixed_cost_amortized': amortized_fixed,
                'distance_to_customer': distance
            },
            timestamp=time.time()
        )
    
    def evaluate_opening_decision(self, potential_customers: List[int]) -> bool:
        """Evaluate whether to open the depot"""
        if self.is_open:
            return True
        
        # Calculate potential revenue from customers
        potential_revenue = 0.0
        for customer_id in potential_customers:
            demand = self.instance.demands.get(customer_id, 0)
            routing_cost = self.instance.travel_costs.get((self.depot_id, customer_id), 0)
            # Assume we can charge a premium over cost
            potential_revenue += routing_cost * 1.2
        
        # Open if potential revenue exceeds fixed cost
        should_open = potential_revenue > self.fixed_cost * 1.1  # 10% margin
        
        if should_open:
            self.is_open = True
            self.operating_cost += self.fixed_cost
            self.update_knowledge('status', 'open')
        
        return should_open
    
    def handle_message(self, message: Message):
        """Handle messages from other agents"""
        if message.message_type == MessageType.ANNOUNCE_DEMAND:
            # Customer announced demand, calculate and send bid
            bid = self.calculate_bid(message.content)
            
            # Send bid back to customer
            response = self.send_message(
                receiver_id=message.sender_id,
                message_type=MessageType.SUBMIT_BID,
                content={'bid': bid.__dict__},
                priority=2
            )
            return response

class VehicleAgent(Agent):
    """Agent representing a vehicle for routing"""
    
    def __init__(self, agent_id: str, vehicle_id: int, instance: LRPInstance):
        super().__init__(agent_id, "vehicle", instance)
        self.vehicle_id = vehicle_id
        self.capacity = instance.vehicle_capacity
        self.current_location = None
        self.assigned_depot = None
        self.route = []
        self.customers_served = []
        self.total_distance = 0.0
        self.total_cost = 0.0
        self.is_available = True
    
    def calculate_route_cost(self, route: List[int]) -> float:
        """Calculate the cost of a route"""
        if len(route) < 2:
            return 0.0
        
        total_cost = 0.0
        for i in range(len(route) - 1):
            u, v = route[i], route[i + 1]
            total_cost += self.instance.travel_costs.get((u, v), float('inf'))
        
        return total_cost
    
    def is_route_feasible(self, route: List[int]) -> bool:
        """Check if a route respects capacity constraints"""
        total_demand = 0.0
        for node in route:
            if node in self.instance.customers:
                total_demand += self.instance.demands.get(node, 0)
        
        return total_demand <= self.capacity
    
    def optimize_route(self, customers: List[int], depot: int) -> List[int]:
        """Optimize route for given customers and depot using nearest neighbor"""
        if not customers:
            return [depot, depot]
        
        route = [depot]
        unvisited = customers.copy()
        
        while unvisited:
            current = route[-1]
            # Find nearest unvisited customer
            nearest = None
            nearest_distance = float('inf')
            
            for customer in unvisited:
                distance = self.instance.travel_costs.get((current, customer), float('inf'))
                if distance < nearest_distance:
                    nearest = customer
                    nearest_distance = distance
            
            # Check capacity constraint
            temp_route = route + [nearest]
            if self.is_route_feasible(temp_route):
                route.append(nearest)
                unvisited.remove(nearest)
            else:
                break
        
        # Return to depot
        route.append(depot)
        
        return route
    
    def handle_message(self, message: Message):
        """Handle messages from other agents"""
        if message.message_type == MessageType.REQUEST_ROUTE:
            # Depot requesting route optimization
            customers = message.content['customers']
            depot = message.content['depot']
            
            if self.is_available:
                route = self.optimize_route(customers, depot)
                route_cost = self.calculate_route_cost(route)
                
                # Send route proposal
                response = self.send_message(
                    receiver_id=message.sender_id,
                    message_type=MessageType.PROPOSE_ROUTE,
                    content={
                        'route': route,
                        'cost': route_cost,
                        'capacity_utilization': sum(self.instance.demands.get(c, 0) for c in route if c in self.instance.customers) / self.capacity
                    },
                    priority=2
                )
                return response

print("Multi-Agent System classes implemented")

In [ ]:
class ContractNetProtocol:
    """Implementation of Contract Net Protocol for agent coordination"""
    
    def __init__(self, instance: LRPInstance):
        self.instance = instance
        self.agents = {}
        self.message_bus = []
        self.contracts = []
        self.auction_history = []
        self.initialize_agents()
    
    def initialize_agents(self):
        """Initialize all agents in the system"""
        
        # Create customer agents
        for customer in self.instance.customers:
            agent = CustomerAgent(f"customer_{customer}", customer, self.instance)
            self.agents[agent.agent_id] = agent
        
        # Create depot agents
        for depot in self.instance.depots:
            agent = DepotAgent(f"depot_{depot}", depot, self.instance)
            self.agents[agent.agent_id] = agent
        
        # Create vehicle agents
        for vehicle in self.instance.vehicles:
            agent = VehicleAgent(f"vehicle_{vehicle}", vehicle, self.instance)
            self.agents[agent.agent_id] = agent
    
    def broadcast_message(self, message: Message):
        """Broadcast a message to all relevant agents"""
        self.message_bus.append(message)
        
        # Deliver message to recipient
        if message.receiver_id in self.agents:
            self.agents[message.receiver_id].receive_message(message)
    
    def run_customer_depot_auction(self):
        """Run auction process between customers and depots"""
        print("\nRunning Customer-Depot Auction...")
        
        customer_agents = [agent for agent in self.agents.values() if agent.agent_type == "customer"]
        depot_agents = [agent for agent in self.agents.values() if agent.agent_type == "depot"]
        
        auction_results = []
        
        # Phase 1: Customers announce demand
        for customer_agent in customer_agents:
            messages = customer_agent.announce_demand(depot_agents)
            for message in messages:
                self.broadcast_message(message)
        
        # Phase 2: Process depot responses and collect bids
        customer_bids = {customer.agent_id: [] for customer in customer_agents}
        
        for depot_agent in depot_agents:
            depot_agent.process_messages()
            
            # Collect outgoing messages (bids)
            outgoing_messages = [msg for msg in self.message_bus if msg.sender_id == depot_agent.agent_id]
            
            for message in outgoing_messages:
                if message.message_type == MessageType.SUBMIT_BID:
                    bid_data = message.content['bid']
                    bid = Bid(
                        agent_id=bid_data['agent_id'],
                        bid_value=bid_data['bid_value'],
                        capacity=bid_data['capacity'],
                        quality_score=bid_data['quality_score'],
                        additional_info=bid_data['additional_info'],
                        timestamp=bid_data['timestamp']
                    )
                    customer_bids[message.receiver_id].append(bid)
        
        # Phase 3: Customers evaluate bids and award contracts
        for customer_agent in customer_agents:
            bids = customer_bids[customer_agent.agent_id]
            
            if bids:
                best_bid = customer_agent.evaluate_bids(bids)
                
                if best_bid:
                    # Award contract to winning depot
                    contract = Contract(
                        contract_id=f"contract_{customer_agent.customer_id}_{int(time.time())}",
                        client_id=customer_agent.agent_id,
                        provider_id=best_bid.agent_id,
                        service_type="depot_service",
                        terms={
                            'customer_id': customer_agent.customer_id,
                            'depot_id': int(best_bid.agent_id.split('_')[1]),
                            'cost': best_bid.bid_value,
                            'quality_score': best_bid.quality_score
                        }
                    )
                    
                    self.contracts.append(contract)
                    
                    # Send award message
                    award_message = customer_agent.send_message(
                        receiver_id=best_bid.agent_id,
                        message_type=MessageType.AWARD_CONTRACT,
                        content={
                            'contract_id': contract.contract_id,
                            'depot_id': contract.terms['depot_id']
                        },
                        priority=3
                    )
                    
                    self.broadcast_message(award_message)
                    
                    # Update depot assignment
                    depot_agent = self.agents[best_bid.agent_id]
                    depot_agent.assigned_customers.append(customer_agent.customer_id)
                    depot_agent.is_open = True
                    
                    auction_results.append({
                        'customer': customer_agent.customer_id,
                        'depot': contract.terms['depot_id'],
                        'cost': best_bid.bid_value,
                        'quality': best_bid.quality_score
                    })
        
        # Process award messages
        for depot_agent in depot_agents:
            depot_agent.process_messages()
        
        return auction_results
    
    def run_vehicle_routing_auction(self):
        """Run vehicle routing auction for each depot"""
        print("\nRunning Vehicle Routing Auction...")
        
        depot_agents = [agent for agent in self.agents.values() if agent.agent_type == "depot" and agent.is_open]
        vehicle_agents = [agent for agent in self.agents.values() if agent.agent_type == "vehicle"]
        
        routing_results = []
        
        for depot_agent in depot_agents:
            if not depot_agent.assigned_customers:
                continue
            
            print(f"Optimizing routes for Depot {depot_agent.depot_id}...")
            
            # Request route proposals from available vehicles
            available_vehicles = [v for v in vehicle_agents if v.is_available]
            
            if not available_vehicles:
                print(f"No available vehicles for Depot {depot_agent.depot_id}")
                continue
            
            # Divide customers among vehicles
            customers_per_vehicle = len(depot_agent.assigned_customers) // len(available_vehicles)
            remaining_customers = depot_agent.assigned_customers.copy()
            
            for i, vehicle_agent in enumerate(available_vehicles):
                if not remaining_customers:
                    break
                
                # Assign customers to this vehicle
                if i < len(available_vehicles) - 1:
                    # Full allocation for all but last vehicle
                    assigned_customers = remaining_customers[:customers_per_vehicle]
                    remaining_customers = remaining_customers[customers_per_vehicle:]
                else:
                    # Remaining customers to last vehicle
                    assigned_customers = remaining_customers
                    remaining_customers = []
                
                if assigned_customers:
                    # Request route from vehicle
                    route_request = depot_agent.send_message(
                        receiver_id=vehicle_agent.agent_id,
                        message_type=MessageType.REQUEST_ROUTE,
                        content={
                            'customers': assigned_customers,
                            'depot': depot_agent.depot_id
                        },
                        priority=2
                    )
                    
                    self.broadcast_message(route_request)
            
            # Process vehicle responses
            for vehicle_agent in available_vehicles:
                vehicle_agent.process_messages()
            
            # Collect route proposals
            route_proposals = []
            for vehicle_agent in available_vehicles:
                outgoing_messages = [msg for msg in self.message_bus if msg.sender_id == vehicle_agent.agent_id]
                
                for message in outgoing_messages:
                    if message.message_type == MessageType.PROPOSE_ROUTE:
                        route_proposals.append({
                            'vehicle_id': vehicle_agent.vehicle_id,
                            'vehicle_agent': vehicle_agent,
                            'route': message.content['route'],
                            'cost': message.content['cost'],
                            'capacity_utilization': message.content['capacity_utilization']
                        })
            
            # Accept route proposals
            for proposal in route_proposals:
                vehicle_agent = proposal['vehicle_agent']
                vehicle_agent.route = proposal['route']
                vehicle_agent.assigned_depot = depot_agent.depot_id
                vehicle_agent.customers_served = [n for n in proposal['route'] if n in self.instance.customers]
                vehicle_agent.total_distance = proposal['cost']
                vehicle_agent.total_cost = proposal['cost']
                vehicle_agent.is_available = False
                
                depot_agent.assigned_vehicles.append(vehicle_agent.vehicle_id)
                
                routing_results.append({
                    'depot': depot_agent.depot_id,
                    'vehicle': vehicle_agent.vehicle_id,
                    'route': proposal['route'],
                    'cost': proposal['cost'],
                    'utilization': proposal['capacity_utilization']
                })
        
        return routing_results
    
    def get_system_solution(self):
        """Extract the final solution from the multi-agent system"""
        
        opened_depots = []
        customer_assignments = {}
        routes = {}
        
        # Get opened depots and assignments
        for agent in self.agents.values():
            if agent.agent_type == "depot" and agent.is_open:
                opened_depots.append(agent.depot_id)
                
                for customer in agent.assigned_customers:
                    customer_assignments[customer] = agent.depot_id
            
            elif agent.agent_type == "vehicle" and not agent.is_available:
                routes[agent.vehicle_id] = agent.route
        
        return {
            'opened_depots': opened_depots,
            'customer_assignments': customer_assignments,
            'routes': routes
        }

print("Contract Net Protocol implemented")

In [ ]:
# Initialize the Multi-Agent System
cnp = ContractNetProtocol(instance)

print(f"Multi-Agent System initialized with {len(cnp.agents)} agents:")
for agent_id, agent in cnp.agents.items():
    print(f"  {agent_id}: {agent.agent_type}")

# Run the complete auction process
print("\n" + "="*60)
print("MULTI-AGENT SYSTEM EXECUTION")
print("="*60)

# Phase 1: Customer-Depot Auction
auction_results = cnp.run_customer_depot_auction()

print("\nCustomer-Depot Assignment Results:")
for result in auction_results:
    print(f"  Customer {result['customer']} → Depot J{result['depot']-3} (Cost: ${result['cost']:.2f}, Quality: {result['quality']:.2f})")

# Phase 2: Vehicle Routing Auction
routing_results = cnp.run_vehicle_routing_auction()

print("\nVehicle Routing Results:")
for result in routing_results:
    print(f"  Vehicle {result['vehicle']}: {result['route']} (Cost: ${result['cost']:.2f}, Utilization: {result['utilization']:.1%})")

# Get final solution
solution = cnp.get_system_solution()

print("\nFinal Multi-Agent Solution:")
print(f"Opened Depots: {solution['opened_depots']}")
print(f"Customer Assignments: {solution['customer_assignments']}")
print(f"Routes: {solution['routes']}")

In [ ]:
def calculate_solution_cost(solution, instance):
    """Calculate total cost of the multi-agent solution"""
    
    if not solution['opened_depots']:
        return float('inf'), 0, 0
    
    # Fixed depot costs
    fixed_cost = sum(instance.depot_costs[j] for j in solution['opened_depots'])
    
    # Routing costs
    routing_cost = 0
    for route in solution['routes'].values():
        for i in range(len(route) - 1):
            u, v = route[i], route[i + 1]
            routing_cost += instance.travel_costs.get((u, v), float('inf'))
    
    total_cost = fixed_cost + routing_cost
    return total_cost, fixed_cost, routing_cost

def analyze_system_performance(cnp, solution):
    """Analyze the performance of the multi-agent system"""
    
    total_cost, fixed_cost, routing_cost = calculate_solution_cost(solution, instance)
    
    # Calculate agent performance metrics
    depot_performance = {}
    vehicle_performance = {}
    
    for agent_id, agent in cnp.agents.items():
        if agent.agent_type == "depot" and agent.is_open:
            depot_performance[agent.depot_id] = {
                'customers_served': len(agent.assigned_customers),
                'total_demand': sum(instance.demands.get(c, 0) for c in agent.assigned_customers),
                'fixed_cost': agent.fixed_cost,
                'vehicles_assigned': len(agent.assigned_vehicles)
            }
        
        elif agent.agent_type == "vehicle" and not agent.is_available:
            vehicle_performance[agent.vehicle_id] = {
                'route_length': len(agent.route),
                'customers_served': len(agent.customers_served),
                'total_distance': agent.total_distance,
                'capacity_utilization': sum(instance.demands.get(c, 0) for c in agent.customers_served) / agent.capacity,
                'assigned_depot': agent.assigned_depot
            }
    
    # System-level metrics
    system_metrics = {
        'total_cost': total_cost,
        'fixed_cost': fixed_cost,
        'routing_cost': routing_cost,
        'depots_opened': len(solution['opened_depots']),
        'vehicles_used': len(solution['routes']),
        'customers_served': len(solution['customer_assignments']),
        'avg_vehicle_utilization': np.mean([v['capacity_utilization'] for v in vehicle_performance.values()]) if vehicle_performance else 0,
        'total_contracts': len(cnp.contracts),
        'successful_auctions': len([c for c in cnp.contracts if c.status == 'pending'])
    }
    
    return system_metrics, depot_performance, vehicle_performance

# Analyze system performance
system_metrics, depot_performance, vehicle_performance = analyze_system_performance(cnp, solution)

print("\n" + "="*60)
print("MULTI-AGENT SYSTEM PERFORMANCE ANALYSIS")
print("="*60)

print(f"\nSystem-Level Metrics:")
for metric, value in system_metrics.items():
    if isinstance(value, float):
        print(f"  {metric.replace('_', ' ').title()}: ${value:.2f}" if 'cost' in metric else f"  {metric.replace('_', ' ').title()}: {value:.2f}")
    else:
        print(f"  {metric.replace('_', ' ').title()}: {value}")

print(f"\nDepot Performance:")
for depot_id, metrics in depot_performance.items():
    print(f"  Depot J{depot_id-3}: {metrics['customers_served']} customers, {metrics['total_demand']:.1f} total demand, {metrics['vehicles_assigned']} vehicles")

print(f"\nVehicle Performance:")
for vehicle_id, metrics in vehicle_performance.items():
    print(f"  Vehicle {vehicle_id}: {metrics['customers_served']} customers, {metrics['capacity_utilization']:.1%} utilization, {metrics['total_distance']:.1f} distance")

In [ ]:
def test_system_resilience(cnp, instance):
    """Test system resilience under various failure scenarios"""
    
    print("\n" + "="*60)
    print("SYSTEM RESILIENCE TESTING")
    print("="*60)
    
    resilience_scenarios = {
        'vehicle_failure': {
            'description': 'Random vehicle failure during operation',
            'test_function': test_vehicle_failure
        },
        'depot_failure': {
            'description': 'Depot agent becomes unavailable',
            'test_function': test_depot_failure
        },
        'communication_delay': {
            'description': 'Communication delays between agents',
            'test_function': test_communication_delay
        }
    }
    
    resilience_results = {}
    
    for scenario_name, scenario_config in resilience_scenarios.items():
        print(f"\nTesting: {scenario_config['description']}")
        
        # Create fresh system for each test
        test_cnp = ContractNetProtocol(instance)
        
        # Run baseline
        baseline_auction = test_cnp.run_customer_depot_auction()
        baseline_routing = test_cnp.run_vehicle_routing_auction()
        baseline_solution = test_cnp.get_system_solution()
        baseline_cost, _, _ = calculate_solution_cost(baseline_solution, instance)
        
        # Apply failure scenario
        scenario_result = scenario_config['test_function'](test_cnp, instance)
        
        # Re-run with failure
        try:
            failure_auction = test_cnp.run_customer_depot_auction()
            failure_routing = test_cnp.run_vehicle_routing_auction()
            failure_solution = test_cnp.get_system_solution()
            failure_cost, _, _ = calculate_solution_cost(failure_solution, instance)
            
            # Calculate resilience metrics
            cost_increase = (failure_cost - baseline_cost) / baseline_cost * 100 if baseline_cost > 0 else float('inf')
            service_degradation = len(baseline_solution['customer_assignments']) - len(failure_solution['customer_assignments'])
            
            resilience_results[scenario_name] = {
                'baseline_cost': baseline_cost,
                'failure_cost': failure_cost,
                'cost_increase_pct': cost_increase,
                'service_degradation': service_degradation,
                'system_recovered': len(failure_solution['customer_assignments']) > 0,
                'scenario_data': scenario_result
            }
            
            print(f"  Baseline Cost: ${baseline_cost:.2f}")
            print(f"  Failure Cost: ${failure_cost:.2f}")
            print(f"  Cost Increase: {cost_increase:.1f}%")
            print(f"  Service Degradation: {service_degradation} customers")
            print(f"  System Recovered: {'Yes' if len(failure_solution['customer_assignments']) > 0 else 'No'}")
            
        except Exception as e:
            resilience_results[scenario_name] = {
                'baseline_cost': baseline_cost,
                'failure_cost': float('inf'),
                'cost_increase_pct': float('inf'),
                'service_degradation': len(baseline_solution['customer_assignments']),
                'system_recovered': False,
                'error': str(e)
            }
            print(f"  System failed to recover: {str(e)}")
    
    return resilience_results

def test_vehicle_failure(cnp, instance):
    """Test vehicle failure scenario"""
    # Randomly disable one vehicle agent
    vehicle_agents = [agent for agent in cnp.agents.values() if agent.agent_type == "vehicle"]
    if vehicle_agents:
        failed_vehicle = random.choice(vehicle_agents)
        failed_vehicle.active = False
        return {'failed_vehicle': failed_vehicle.vehicle_id}
    return {}

def test_depot_failure(cnp, instance):
    """Test depot failure scenario"""
    # Disable one depot agent
    depot_agents = [agent for agent in cnp.agents.values() if agent.agent_type == "depot"]
    if depot_agents:
        failed_depot = random.choice(depot_agents)
        failed_depot.active = False
        return {'failed_depot': failed_depot.depot_id}
    return {}

def test_communication_delay(cnp, instance):
    """Test communication delay scenario"""
    # Simulate communication delays by adding latency to message processing
    original_process = Agent.process_messages
    
    def delayed_process_messages(self):
        time.sleep(0.01)  # 10ms delay
        return original_process(self)
    
    # Apply delay to all agents
    for agent in cnp.agents.values():
        agent.process_messages = delayed_process_messages.__get__(agent, Agent)
    
    return {'communication_delay_ms': 10}

# Run resilience testing
resilience_results = test_system_resilience(cnp, instance)

def visualize_resilience_results(resilience_results):
    """Visualize resilience testing results"""
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    
    scenarios = list(resilience_results.keys())
    baseline_costs = [resilience_results[s]['baseline_cost'] for s in scenarios]
    failure_costs = [resilience_results[s]['failure_cost'] if resilience_results[s]['failure_cost'] != float('inf') else 0 for s in scenarios]
    cost_increases = [resilience_results[s]['cost_increase_pct'] if resilience_results[s]['cost_increase_pct'] != float('inf') else 100 for s in scenarios]
    service_degradations = [resilience_results[s]['service_degradation'] for s in scenarios]
    
    # Plot 1: Cost comparison
    x = np.arange(len(scenarios))
    width = 0.35
    
    ax1.bar(x - width/2, baseline_costs, width, label='Baseline', alpha=0.7)
    ax1.bar(x + width/2, failure_costs, width, label='With Failure', alpha=0.7)
    ax1.set_xlabel('Failure Scenario')
    ax1.set_ylabel('Total Cost ($)')
    ax1.set_title('Cost Impact of Failures')
    ax1.set_xticks(x)
    ax1.set_xticklabels([s.replace('_', ' ').title() for s in scenarios])
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Cost increase percentage
    bars = ax2.bar(scenarios, cost_increases, alpha=0.7, color='orange')
    ax2.set_ylabel('Cost Increase (%)')
    ax2.set_title('Relative Cost Increase')
    ax2.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bar, value in zip(bars, cost_increases):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 1,
                f'{value:.1f}%', ha='center', va='bottom')
    
    # Plot 3: Service degradation
    bars = ax3.bar(scenarios, service_degradations, alpha=0.7, color='red')
    ax3.set_ylabel('Customers Lost')
    ax3.set_title('Service Degradation')
    ax3.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bar, value in zip(bars, service_degradations):
        height = bar.get_height()
        if height > 0:
            ax3.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                    f'{int(value)}', ha='center', va='bottom')
    
    # Plot 4: Recovery status
    recovery_status = [1 if resilience_results[s]['system_recovered'] else 0 for s in scenarios]
    colors = ['green' if status == 1 else 'red' for status in recovery_status]
    bars = ax4.bar(scenarios, recovery_status, alpha=0.7, color=colors)
    ax4.set_ylabel('Recovery Status (1=Success, 0=Failure)')
    ax4.set_title('System Recovery Capability')
    ax4.set_ylim(0, 1.1)
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Visualize resilience results
visualize_resilience_results(resilience_results)

In [ ]:
def visualize_multi_agent_solution(solution, instance, cost):
    """Visualize the multi-agent system solution"""
    
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    
    # Define positions
    positions = {
        1: (2, 8), 2: (4, 6), 3: (8, 7),  # Customers
        4: (3, 3), 5: (7, 3)              # Depots
    }
    
    # Plot nodes
    for node, pos in positions.items():
        if node in instance.customers:
            ax.scatter(pos[0], pos[1], s=300, c='lightblue', 
                      edgecolors='navy', linewidth=2, zorder=5)
            ax.annotate(f'C{node}\n(d={instance.demands[node]})', 
                       xy=pos, xytext=(pos[0]+0.3, pos[1]+0.3),
                       fontsize=10, fontweight='bold')
        elif node in instance.depots:
            color = 'lightgreen' if node in solution['opened_depots'] else 'lightgray'
            ax.scatter(pos[0], pos[1], s=400, c=color, 
                      edgecolors='darkgreen', linewidth=2, zorder=5)
            status = "OPEN" if node in solution['opened_depots'] else "CLOSED"
            cost = instance.depot_costs[node]
            ax.annotate(f'J{node-3}\n({status})\n(${cost})', 
                       xy=pos, xytext=(pos[0]-0.8, pos[1]-1.2),
                       fontsize=10, fontweight='bold')
    
    # Plot routes
    colors = ['red', 'blue', 'green', 'orange']
    for k, route in solution['routes'].items():
        if len(route) > 1:
            color = colors[k % len(colors)]
            for i in range(len(route) - 1):
                u, v = route[i], route[i + 1]
                u_pos, v_pos = positions[u], positions[v]
                ax.plot([u_pos[0], v_pos[0]], [u_pos[1], v_pos[1]], 
                       color=color, linewidth=2, alpha=0.7, 
                       label=f'Vehicle {k}' if i == 0 else '')
                
                # Add arrow
                mid_x = (u_pos[0] + v_pos[0]) / 2
                mid_y = (u_pos[1] + v_pos[1]) / 2
                dx = v_pos[0] - u_pos[0]
                dy = v_pos[1] - u_pos[1]
                ax.annotate('', xy=(mid_x + dx*0.1, mid_y + dy*0.1), 
                           xytext=(mid_x - dx*0.1, mid_y - dy*0.1),
                           arrowprops=dict(arrowstyle='->', color=color, lw=1.5))
    
    # Plot assignments
    for customer, depot in solution['customer_assignments'].items():
        if depot in solution['opened_depots']:
            cust_pos = positions[customer]
            depot_pos = positions[depot]
            ax.plot([cust_pos[0], depot_pos[0]], [cust_pos[1], depot_pos[1]], 
                   'k--', alpha=0.3, linewidth=1)
    
    ax.set_xlim(-1, 10)
    ax.set_ylim(0, 10)
    ax.set_xlabel('X Coordinate', fontsize=12)
    ax.set_ylabel('Y Coordinate', fontsize=12)
    ax.set_title(f'Multi-Agent System Solution\nTotal Cost: ${cost:.2f}', 
                 fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right')
    
    plt.tight_layout()
    plt.show()

# Visualize the multi-agent solution
total_cost, _, _ = calculate_solution_cost(solution, instance)
visualize_multi_agent_solution(solution, instance, total_cost)

### Why this Tier exists vs earlier Tiers
The Multi-Agent System addresses fundamental limitations of centralized optimization approaches:

**Tier 1-5 Limitations:**
- ❌ Centralized control - single point of failure
- ❌ No autonomy or distributed intelligence
- ❌ Limited scalability with centralized decision-making
- ❌ No resilience to individual component failures
- ❌ Static optimization - no adaptive behavior

**Tier 6 Advantages:**
- ✅ **Distributed intelligence** - autonomous decision-making at agent level
- ✅ **Emergent optimization** - solutions emerge from agent interactions
- ✅ **System resilience** - no single point of failure
- ✅ **Scalable architecture** - agents can be added/removed dynamically
- ✅ **Adaptive behavior** - agents learn and adapt over time
- ✅ **Contract Net Protocol** - structured agent coordination

### Pros / Cons vs earlier Tiers

**Pros:**
- ✅ **Fault tolerance** - system continues operating with agent failures
- ✅ **Scalability** - easy to add new agents without redesign
- ✅ **Flexibility** - agents can have different strategies and capabilities
- ✅ **Real-time adaptation** - agents respond to changes immediately
- ✅ **Robustness** - distributed decision-making reduces vulnerability
- ✅ **Natural parallelism** - agents operate concurrently

**Cons:**
- ❌ **Coordination overhead** - communication between agents adds complexity
- ❌ **No global optimality guarantee** - emergent solutions may be suboptimal
- ❌ **Complex debugging** - distributed behavior is harder to analyze
- ❌ **Communication dependencies** - system performance depends on network reliability
- ❌ **Potential for deadlocks** - agent negotiations may get stuck

### When to use this Tier
- **Large-scale distributed systems** with many decision points
- **High-availability environments** requiring fault tolerance
- **Dynamic markets** with rapidly changing conditions
- **Multi-stakeholder environments** with different objectives
- **Real-time systems** requiring immediate adaptation
- **Complex ecosystems** where centralized control is impractical

### Key Multi-Agent System Innovations

1. **Contract Net Protocol**: Structured negotiation and coordination framework

2. **Autonomous Agents**: Independent decision-making with local knowledge

3. **Auction Mechanisms**: Market-based resource allocation

4. **Emergent Behavior**: Complex solutions from simple agent interactions

5. **Fault Tolerance**: Graceful degradation under component failures

6. **Distributed Intelligence**: Collective problem-solving without central control

The Multi-Agent System represents a paradigm shift from centralized optimization to distributed intelligence, where optimal solutions emerge from the coordinated interactions of autonomous agents, creating a resilient and adaptive supply chain ecosystem.